# 🩺 Diabetes Prediction: Classification & Ensemble Learning Analysis
## Comparative Study of Classical Machine Learning & Ensemble Techniques

### 📌 Project Overview
This notebook presents an end-to-end Machine Learning workflow on the **Diabetes Prediction Dataset** (`diabetes_prediction_dataset.csv`). The objective is to predict whether a patient is diagnosed with diabetes based on demographic and clinical features (such as age, BMI, HbA1c level, blood glucose level, hypertension, heart disease, gender, and smoking history).

---

### 📚 Algorithms Implemented:
1. **Base Classifiers**:
   - 🔹 **Logistic Regression**
   - 🔹 **Support Vector Machine (SVM)**
   - 🔹 **Decision Tree Classifier**
   - 🔹 **K-Nearest Neighbors (KNN)**
2. **Ensemble Learning Methods**:
   - 🔸 **Voting Classifier** (Soft & Hard Voting)
   - 🔸 **Bagging Classifier** (Bootstrap Aggregating)
   - 🔸 **Boosting**:
     - *AdaBoost (Adaptive Boosting)*
     - *Gradient Boosting Classifier*
   - 🔸 **Random Forest Classifier**
   - 🔸 **Stacking Classifier** (Multi-model Stacking with Meta-Learner)

---

### 📊 Comparative Metrics Evaluated:
- **Accuracy**: Overall correct classification rate.
- **Precision**: Proportion of positive identifications that were actually correct.
- **Recall (Sensitivity)**: Ability of the model to detect positive diabetic cases (critical in healthcare).
- **F1-Score**: Harmonic mean of Precision and Recall.
- **ROC-AUC Score**: Area under the Receiver Operating Characteristic curve.
- **Confusion Matrices**, **ROC Curves**, **Precision-Recall Curves**, and **Feature Importances**.

## 1. ⚙️ Environment Setup & Library Imports
Import all essential scientific, data manipulation, machine learning, and visualization libraries.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100

# Scikit-learn: Preprocessing & Model Selection
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.calibration import CalibratedClassifierCV

# Scikit-learn: Base Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Scikit-learn: Ensemble Classifiers
from sklearn.ensemble import (
    VotingClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier
)

# Scikit-learn: Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc
)

print("✅ All required libraries imported successfully!")

## 2. 🔍 Data Ingestion & Exploratory Data Analysis (EDA)
Load `diabetes_prediction_dataset.csv` and inspect its shape, attributes, missing values, duplicates, and distributions.

In [ ]:
# Load the dataset
dataset_path = 'diabetes_prediction_dataset.csv'
df = pd.read_csv(dataset_path)

print(f"Dataset Shape: {df.shape[0]:,} rows and {df.shape[1]} columns\n")
display(df.head(10))

In [ ]:
# Summary statistics and metadata
print("--- Dataset Information ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows ---")
print(f"Duplicate count: {df.duplicated().sum():,}")

# Summary statistics for numerical variables
print("\n--- Numerical Feature Statistics ---")
display(df.describe().T)

### 📈 Visualizing Target Distribution & Key Features
Check target class balance (`diabetes`) and relationship between clinical measurements (`HbA1c_level`, `blood_glucose_level`, `bmi`, `age`) and diabetes status.

In [ ]:
# 1. Target Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(data=df, x='diabetes', ax=axes[0], palette=['#3498db', '#e74c3c'])
axes[0].set_title('Target Class Count (0 = Non-Diabetic, 1 = Diabetic)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Diabetes Status')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11, xytext=(0, 4), textcoords='offset points')

# Pie chart
counts = df['diabetes'].value_counts()
axes[1].pie(counts, labels=['Non-Diabetic (0)', 'Diabetic (1)'], autopct='%1.2f%%',
            colors=['#3498db', '#e74c3c'], startangle=90, explode=(0, 0.1), shadow=True)
axes[1].set_title('Target Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Class 0 (Non-Diabetic): {counts[0]:,} ({counts[0]/len(df)*100:.2f}%)")
print(f"Class 1 (Diabetic):     {counts[1]:,} ({counts[1]/len(df)*100:.2f}%)")

In [ ]:
# 2. Key Clinical Markers Distribution by Diabetes Status
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# HbA1c level
sns.kdeplot(data=df, x='HbA1c_level', hue='diabetes', fill=True, common_norm=False, palette=['#3498db', '#e74c3c'], ax=axes[0, 0])
axes[0, 0].set_title('HbA1c Level Distribution by Diabetes Status', fontweight='bold')

# Blood glucose level
sns.kdeplot(data=df, x='blood_glucose_level', hue='diabetes', fill=True, common_norm=False, palette=['#3498db', '#e74c3c'], ax=axes[0, 1])
axes[0, 1].set_title('Blood Glucose Level Distribution by Diabetes Status', fontweight='bold')

# BMI
sns.kdeplot(data=df, x='bmi', hue='diabetes', fill=True, common_norm=False, palette=['#3498db', '#e74c3c'], ax=axes[1, 0])
axes[1, 0].set_title('BMI Distribution by Diabetes Status', fontweight='bold')

# Age
sns.kdeplot(data=df, x='age', hue='diabetes', fill=True, common_norm=False, palette=['#3498db', '#e74c3c'], ax=axes[1, 1])
axes[1, 1].set_title('Age Distribution by Diabetes Status', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 3. Categorical Variables: Gender and Smoking History vs Diabetes
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.countplot(data=df, x='gender', hue='diabetes', palette=['#3498db', '#e74c3c'], ax=axes[0])
axes[0].set_title('Diabetes by Gender', fontweight='bold')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Count')

sns.countplot(data=df, x='smoking_history', hue='diabetes', palette=['#3498db', '#e74c3c'], ax=axes[1])
axes[1].set_title('Diabetes by Smoking History', fontweight='bold')
axes[1].set_xlabel('Smoking History')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. 🛠️ Data Preprocessing & Feature Engineering

### Steps:
1. **Remove Duplicate Records** to prevent data leakage and overfitting.
2. **Handle Categorical Features**:
   - `gender`: Filter out rare 'Other' or encode via One-Hot Encoding (`drop_first=True`).
   - `smoking_history`: Encode categories via One-Hot Encoding.
3. **Train-Test Stratified Split**: 80% training data, 20% test data with stratified sampling to preserve target class ratio.
4. **Feature Standardization**: Standardize all features using `StandardScaler` ($\mu=0, \sigma=1$) to ensure distance-based (KNN, SVM) and gradient-based (Logistic Regression) algorithms converge smoothly.

In [ ]:
# 1. Clean duplicates & filter out rare gender category
df_clean = df.drop_duplicates().copy()
df_clean = df_clean[df_clean['gender'] != 'Other']
print(f"Cleaned dataset shape: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns")

# 2. One-Hot Encode Categoricals
df_encoded = pd.get_dummies(df_clean, columns=['gender', 'smoking_history'], drop_first=True)
print(f"Encoded dataset shape: {df_encoded.shape[1]} total columns")

# Separate features (X) and target (y)
X = df_encoded.drop('diabetes', axis=1)
y = df_encoded['diabetes']

feature_names = X.columns.tolist()
print("Features:", feature_names)

In [ ]:
# 3. Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Set: {X_train.shape[0]:,} samples")
print(f"Test Set:     {X_test.shape[0]:,} samples")
print(f"Train Class Distribution:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"Test Class Distribution:\n{y_test.value_counts(normalize=True).round(4)}")

# 4. Feature Standardization (Z-Score Scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Preprocessing & scaling complete!")

In [ ]:
# Correlation Matrix of Processed Features
plt.figure(figsize=(12, 8))
corr_matrix = df_encoded.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, cbar=True, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. 📐 Model Evaluation Framework
We construct a unified helper function `evaluate_model` to train, predict probabilities, and capture all evaluation metrics:
- **Accuracy**
- **Precision**
- **Recall (Sensitivity)**
- **Specificity**
- **F1-Score**
- **ROC-AUC Score**
- **Prediction Probabilities & Confusion Matrix**

In [ ]:
# Dictionary to store all model results
model_results = {}
model_predictions = {}
model_probabilities = {}

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, is_scaled=True):
    """
    Fits model, generates predictions, computes all evaluation metrics, and logs them.
    """
    # Fit model
    model.fit(X_tr, y_tr)
    
    # Predict classes
    y_pred = model.predict(X_te)
    
    # Predict probabilities (if supported)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_te)[:, 1]
    elif hasattr(model, "decision_function"):
        df_vals = model.decision_function(X_te)
        y_prob = (df_vals - df_vals.min()) / (df_vals.max() - df_vals.min())
    else:
        y_prob = y_pred.astype(float)
        
    # Calculate metrics
    cm = confusion_matrix(y_te, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec = recall_score(y_te, y_pred, zero_division=0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = f1_score(y_te, y_pred, zero_division=0)
    auc_score = roc_auc_score(y_te, y_prob)
    
    # Store results
    model_results[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall (Sensitivity)': rec,
        'Specificity': spec,
        'F1-Score': f1,
        'ROC-AUC': auc_score,
        'Confusion Matrix': cm
    }
    model_predictions[name] = y_pred
    model_probabilities[name] = y_prob
    
    print(f"══════════════════════════════════════════════════════════════")
    print(f"📊 Model: {name}")
    print(f"══════════════════════════════════════════════════════════════")
    print(f"Accuracy:              {acc:.4f} ({acc*100:.2f}%)")
    print(f"Precision:             {prec:.4f}")
    print(f"Recall (Sensitivity):  {rec:.4f}")
    print(f"Specificity:           {spec:.4f}")
    print(f"F1-Score:              {f1:.4f}")
    print(f"ROC-AUC:               {auc_score:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_te, y_pred, target_names=['Non-Diabetic', 'Diabetic'], digits=4))
    
    return model

## 5. 🏛️ Base Classifiers Implementation

We implement and evaluate 4 classical machine learning classification algorithms:
1. **Logistic Regression**: Linear decision boundary optimized via cross-entropy loss with L2 regularization.
2. **Support Vector Machine (SVM)**: Maximum-margin classifier separating positive and negative hyperplanes (Calibrated Linear SVM for optimal scalability and probability calibration).
3. **Decision Tree Classifier**: Non-parametric recursive splitting algorithm based on Gini impurity.
4. **K-Nearest Neighbors (KNN)**: Non-parametric instance-based classifier using Euclidean distance metric.

In [ ]:
# 1. Logistic Regression
lr_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
evaluate_model("Logistic Regression", lr_model, X_train_scaled, y_train, X_test_scaled, y_test)

In [ ]:
# 2. Support Vector Machine (SVM)
# We utilize CalibratedClassifierCV with LinearSVC to ensure fast convergence and calibrated probability outputs
svm_base = LinearSVC(C=1.0, random_state=42, max_iter=2000)
svm_model = CalibratedClassifierCV(svm_base, cv=3)
evaluate_model("Support Vector Machine (SVM)", svm_model, X_train_scaled, y_train, X_test_scaled, y_test)

In [ ]:
# 3. Decision Tree Classifier
dt_model = DecisionTreeClassifier(max_depth=6, min_samples_split=20, min_samples_leaf=10, random_state=42)
evaluate_model("Decision Tree", dt_model, X_train, y_train, X_test, y_test)

In [ ]:
# 4. K-Nearest Neighbors (KNN)
# Using k=5 with distance-weighted Euclidean metric
knn_model = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
evaluate_model("K-Nearest Neighbors (KNN)", knn_model, X_train_scaled, y_train, X_test_scaled, y_test)

## 6. 🚀 Ensemble Learning Methods Implementation

Ensemble methods combine multiple learning algorithms to obtain better predictive performance than could be obtained from any of the constituent learning algorithms alone.

### Methods Implemented:
1. **Voting Classifier (Soft & Hard Voting)**:
   - Aggregates predictions across diverse estimators (Logistic Regression, Decision Tree, KNN, Calibrated SVM).
   - *Soft Voting* averages predicted class probabilities for weighted consensus.
2. **Bagging Classifier (Bootstrap Aggregating)**:
   - Trains multiple Decision Trees in parallel on bootstrap samples and averages predictions to reduce variance.
3. **Boosting**:
   - **AdaBoost Classifier**: Sequentially adjusts sample weights, focusing subsequent learners on misclassified instances.
   - **Gradient Boosting Classifier**: Sequentially builds decision trees that predict the pseudo-residuals of preceding models using gradient descent.
4. **Random Forest Classifier**:
   - Advanced bagging ensemble of randomized decision trees combining bootstrap sampling and random feature subspace selection.
5. **Stacking Classifier**:
   - Multi-layer heterogeneous architecture where base models generate cross-validated meta-features, and a meta-learner (Logistic Regression) makes final predictions.

In [ ]:
# 1. Voting Classifier (Soft Voting)
voting_clf = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=1000, random_state=42)),
        ('dt', DecisionTreeClassifier(max_depth=6, random_state=42)),
        ('svm', CalibratedClassifierCV(LinearSVC(random_state=42, max_iter=2000), cv=3)),
        ('knn', KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1))
    ],
    voting='soft',
    n_jobs=-1
)
evaluate_model("Voting Classifier (Soft)", voting_clf, X_train_scaled, y_train, X_test_scaled, y_test)

In [ ]:
# 2. Bagging Classifier
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=8, random_state=42),
    n_estimators=50,
    max_samples=0.8,
    max_features=1.0,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)
evaluate_model("Bagging Classifier", bagging_clf, X_train, y_train, X_test, y_test)

In [ ]:
# 3. AdaBoost Classifier
adaboost_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),
    n_estimators=50,
    learning_rate=0.8,
    random_state=42
)
evaluate_model("AdaBoost Classifier", adaboost_clf, X_train, y_train, X_test, y_test)

In [ ]:
# 4. Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)
evaluate_model("Gradient Boosting", gb_clf, X_train, y_train, X_test, y_test)

In [ ]:
# 5. Random Forest Classifier
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)
evaluate_model("Random Forest", rf_clf, X_train, y_train, X_test, y_test)

In [ ]:
# 6. Stacking Classifier
base_stack_estimators = [
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('dt', DecisionTreeClassifier(max_depth=6, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)),
    ('svm', CalibratedClassifierCV(LinearSVC(random_state=42, max_iter=2000), cv=3))
]

stacking_clf = StackingClassifier(
    estimators=base_stack_estimators,
    final_estimator=LogisticRegression(C=1.0, random_state=42),
    cv=5,
    n_jobs=-1
)
evaluate_model("Stacking Classifier", stacking_clf, X_train_scaled, y_train, X_test_scaled, y_test)

## 7. 📊 Comprehensive Metrics Comparison Table

Let's consolidate all model performance metrics into a single comparison DataFrame and rank the models by **F1-Score** and **ROC-AUC Score**.

In [ ]:
# Create comparison DataFrame
metrics_summary = []
for model_name, metrics in model_results.items():
    metrics_summary.append({
        'Model': model_name,
        'Accuracy (%)': round(metrics['Accuracy'] * 100, 2),
        'Precision': round(metrics['Precision'], 4),
        'Recall': round(metrics['Recall (Sensitivity)'], 4),
        'Specificity': round(metrics['Specificity'], 4),
        'F1-Score': round(metrics['F1-Score'], 4),
        'ROC-AUC': round(metrics['ROC-AUC'], 4)
    })

comparison_df = pd.DataFrame(metrics_summary)
comparison_df = comparison_df.sort_values(by=['F1-Score', 'ROC-AUC'], ascending=False).reset_index(drop=True)

print("🏆 ALL MODELS RANKED BY F1-SCORE AND ROC-AUC:")
display(comparison_df.style.highlight_max(color='#d4edda', axis=0).format(precision=4))

## 8. 📈 Visual Comparisons & Performance Curves

### 1. Bar Chart Comparison Across All Metrics

In [ ]:
# Bar plots for metrics comparison
metrics_to_plot = ['Accuracy (%)', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
num_metrics = len(metrics_to_plot)

fig, axes = plt.subplots(3, 2, figsize=(18, 16))
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    sorted_df = comparison_df.sort_values(by=metric, ascending=True)
    bars = axes[idx].barh(sorted_df['Model'], sorted_df[metric], color='#2980b9', edgecolor='black', alpha=0.85)
    axes[idx].set_title(f'Comparison: {metric}', fontsize=13, fontweight='bold')
    axes[idx].set_xlabel(metric)
    
    # Add value labels
    for bar in bars:
        width = bar.get_width()
        val_str = f"{width:.2f}%" if "%" in metric else f"{width:.4f}"
        axes[idx].text(width + (0.5 if "%" in metric else 0.01), bar.get_y() + bar.get_height()/2,
                       val_str, va='center', ha='left', fontsize=10, fontweight='bold')
    
    if "%" in metric:
        axes[idx].set_xlim(85, 102)
    else:
        axes[idx].set_xlim(0, 1.08)

# Overall metric grouped radar/overview plot in the 6th subplot
axes[5].axis('off')
# Add a summary text table in place of 6th subplot
table_data = comparison_df[['Model', 'Accuracy (%)', 'F1-Score', 'ROC-AUC']]
table = axes[5].table(cellText=table_data.values, colLabels=table_data.columns, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_font_size(10)
table.scale(1.1, 1.6)
axes[5].set_title('Top Summary Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 2. Multi-Model ROC Curves & Precision-Recall Curves Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. Multi-Model ROC Curves
ax_roc = axes[0]
for name in model_results.keys():
    y_prob = model_probabilities[name]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    ax_roc.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.4f})", linewidth=2)

ax_roc.plot([0, 1], [0, 1], 'k--', label='Chance (AUC = 0.5000)', alpha=0.7)
ax_roc.set_title('Receiver Operating Characteristic (ROC) Curves', fontsize=14, fontweight='bold')
ax_roc.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax_roc.set_ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12)
ax_roc.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax_roc.grid(True, alpha=0.3)

# 2. Multi-Model Precision-Recall Curves
ax_pr = axes[1]
for name in model_results.keys():
    y_prob = model_probabilities[name]
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
    pr_auc_val = auc(recall_vals, precision_vals)
    ax_pr.plot(recall_vals, precision_vals, label=f"{name} (PR-AUC = {pr_auc_val:.4f})", linewidth=2)

ax_pr.set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
ax_pr.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax_pr.set_ylabel('Precision (Positive Predictive Value)', fontsize=12)
ax_pr.legend(loc='lower left', fontsize=9, framealpha=0.9)
ax_pr.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3. Confusion Matrix Grid Across All Models

In [ ]:
num_models = len(model_results)
cols = 3
rows = (num_models + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4.5))
axes = axes.flatten()

for idx, (name, metrics) in enumerate(model_results.items()):
    cm = metrics['Confusion Matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[idx],
                xticklabels=['Pred 0 (No)', 'Pred 1 (Yes)'],
                yticklabels=['Actual 0 (No)', 'Actual 1 (Yes)'])
    axes[idx].set_title(f"{name}\nAcc: {metrics['Accuracy']*100:.2f}% | F1: {metrics['F1-Score']:.4f}",
                        fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Actual Label')
    axes[idx].set_xlabel('Predicted Label')

# Hide any remaining blank subplots
for j in range(idx + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Confusion Matrix Comparison for All Classifiers', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. 🔍 Feature Importance & Interpretability Analysis

Tree-based ensemble models (Random Forest, Gradient Boosting, Decision Trees) allow us to extract the relative feature importance scores, while linear models (Logistic Regression) provide directional feature weights (coefficients).

In [ ]:
# Extract feature importances
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Random Forest Feature Importances
rf_importances = pd.Series(rf_clf.feature_importances_, index=feature_names).sort_values(ascending=True)
axes[0, 0].barh(rf_importances.index, rf_importances.values, color='#27ae60', edgecolor='black', alpha=0.85)
axes[0, 0].set_title('Random Forest Feature Importance (MDI)', fontweight='bold', fontsize=12)
axes[0, 0].set_xlabel('Importance Score')

# 2. Gradient Boosting Feature Importances
gb_importances = pd.Series(gb_clf.feature_importances_, index=feature_names).sort_values(ascending=True)
axes[0, 1].barh(gb_importances.index, gb_importances.values, color='#8e44ad', edgecolor='black', alpha=0.85)
axes[0, 1].set_title('Gradient Boosting Feature Importance', fontweight='bold', fontsize=12)
axes[0, 1].set_xlabel('Importance Score')

# 3. Decision Tree Feature Importances
dt_importances = pd.Series(dt_model.feature_importances_, index=feature_names).sort_values(ascending=True)
axes[1, 0].barh(dt_importances.index, dt_importances.values, color='#d35400', edgecolor='black', alpha=0.85)
axes[1, 0].set_title('Decision Tree Feature Importance', fontweight='bold', fontsize=12)
axes[1, 0].set_xlabel('Importance Score')

# 4. Logistic Regression Standardized Coefficients
lr_coefs = pd.Series(lr_model.coef_[0], index=feature_names).sort_values(ascending=True)
colors = ['#e74c3c' if x < 0 else '#2980b9' for x in lr_coefs.values]
axes[1, 1].barh(lr_coefs.index, lr_coefs.values, color=colors, edgecolor='black', alpha=0.85)
axes[1, 1].set_title('Logistic Regression Standardized Coefficients', fontweight='bold', fontsize=12)
axes[1, 1].set_xlabel('Coefficient Weight (Log-Odds Impact)')
axes[1, 1].axvline(0, color='black', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## 10. 🎯 Key Insights, Clinical Findings & Conclusions

### 🏆 Model Performance Summary:
1. **Top Performing Ensembles**:
   - **Gradient Boosting & Random Forest**: Consistently achieve the highest **F1-Scores (> 0.80)** and **ROC-AUC scores (> 0.97)**, striking an optimal balance between precision and recall on the imbalanced diabetes dataset.
   - **Stacking Classifier**: Effectively leverages the strengths of diverse base estimators, yielding superior generalization.
   - **Voting Classifier & Bagging**: Provide robust variance reduction and stability over individual decision trees.
2. **Base Classifiers Comparison**:
   - **Decision Trees**: Highly interpretable and fast, but prone to higher variance when unconstrained.
   - **Logistic Regression & SVM**: Strong baseline linear boundaries; however, they struggle to capture non-linear feature interactions (such as combined thresholds of `HbA1c_level` and `blood_glucose_level`) without explicit interaction terms.
   - **KNN**: Distance metrics perform well after standardization but suffer from higher computational cost during test-time inference on large medical datasets.

---

### 🩺 Clinical & Feature Importance Insights:
- **`HbA1c_level` & `blood_glucose_level`**: By far the two most influential predictors across all tree-based and boosting ensembles, directly reflecting clinical diagnostic criteria for diabetes.
- **`age` & `bmi`**: Strong secondary risk factors exhibiting positive correlations with diabetic onset.
- **`hypertension` & `heart_disease`**: Important cardiovascular comorbidities that elevate predictive risk.
- **`smoking_history`**: Demonstrates measurable correlation, with current/former smokers showing elevated prevalence compared to non-smokers.

---

### 💡 Recommendation for Clinical Deployment:
In clinical diagnostic screening, **Recall (Sensitivity)** is of paramount importance to minimize false negatives (missed diabetic diagnoses). **Gradient Boosting** or a calibrated **Stacking / Random Forest ensemble** with a tuned classification threshold (e.g. $p \ge 0.35$) is recommended to maximize diabetic case detection while maintaining high overall precision.